# 03. Feature Engineering

## 목표
화학식(formula)을 ML이 학습할 수 있는 숫자 feature로 변환한다.

**핵심 아이디어:** 소재의 조성(어떤 원소가 몇 개 있는지)에서  
원자 수준의 물리·화학적 성질(전기음성도, 이온 반지름, 이온화에너지 등)을 추출 →  
이 feature들이 밴드갭과 어떤 관계인지 XGBoost로 학습.

**무기화학 관점:** 밴드갭은 양이온의 d/f-orbital 점유, 전기음성도 차이(이온결합성), 이온 반지름(결정 구조)에 의해 결정됨.

In [ ]:
import pandas as pd
import numpy as np
import ast
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty

df = pd.read_csv('../data/highk_candidates.csv')
df['elements'] = df['elements'].apply(ast.literal_eval)
print('로드 완료:', df.shape)
df.head(3)

## 1. 화학식 → Composition 객체 변환

In [ ]:
# pymatgen Composition 객체로 변환 (matminer featurizer 입력값)
def safe_composition(formula):
    try:
        return Composition(formula)
    except:
        return None

df['composition'] = df['formula'].apply(safe_composition)
df = df[df['composition'].notna()].copy()
print('변환 완료:', len(df), '개')

## 2. Magpie Feature 추출

Magpie(Materials Agnostic Platform for Informatics and Exploration)에서 제공하는  
원소별 물성값의 통계(평균, 표준편차, 최솟값, 최댓값, 범위)를 feature로 사용.

**추출되는 물성:**
- `AtomicWeight` — 원자량
- `Column`, `Row` — 주기율표 위치 (족, 주기)
- `Electronegativity` — 폴링 전기음성도 → 이온결합성, 밴드갭과 직결
- `NsValence`, `NpValence`, `NdValence` — s/p/d 최외각 전자 수 → 오비탈 기여
- `GSbandgap` — 원소 자체의 밴드갭 참고값
- `MeltingT` — 녹는점 → 결합 세기 간접 지표

In [ ]:
featurizer = ElementProperty.from_preset('magpie')

print('추출할 feature 수:', len(featurizer.feature_labels()))
print('샘플 feature명:', featurizer.feature_labels()[:6])

In [ ]:
# feature 추출 (3천 개라 1~2분 소요)
featurizer.set_n_jobs(1)
df_feat = featurizer.featurize_dataframe(df, col_id='composition', ignore_errors=True)
print('Feature 추출 완료:', df_feat.shape)

## 3. 결측값 처리

In [ ]:
feature_cols = featurizer.feature_labels()

# NaN 비율 확인
nan_ratio = df_feat[feature_cols].isnull().mean()
print('NaN 많은 feature Top 10:')
print(nan_ratio[nan_ratio > 0].sort_values(ascending=False).head(10))

# NaN 50% 이상인 feature 제거
drop_cols = nan_ratio[nan_ratio > 0.5].index.tolist()
print(f'\n제거할 feature: {len(drop_cols)}개')
feature_cols = [c for c in feature_cols if c not in drop_cols]

# 나머지 NaN은 중앙값으로 채우기
df_feat[feature_cols] = df_feat[feature_cols].fillna(df_feat[feature_cols].median())
print(f'최종 feature 수: {len(feature_cols)}')

## 4. 기본 상관관계 확인 — 어떤 feature가 밴드갭과 관련 있나

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

corr = df_feat[feature_cols + ['band_gap']].corr()['band_gap'].drop('band_gap')
top_corr = corr.abs().sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 5))
colors = ['tomato' if corr[i] < 0 else 'steelblue' for i in top_corr.index]
plt.barh(top_corr.index[::-1], corr[top_corr.index[::-1]], color=colors[::-1])
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Correlation with Band Gap')
plt.title('Top 15 Features Correlated with Band Gap')
plt.tight_layout()
plt.savefig('../data/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n상위 상관 feature:')
print(corr[top_corr.index].round(3))

## 5. 최종 데이터셋 저장

In [ ]:
# 저장할 컬럼: 식별자 + feature + 타겟
save_cols = ['material_id', 'formula', 'band_gap'] + feature_cols
df_final = df_feat[save_cols].copy()

df_final.to_csv('../data/featurized_highk.csv', index=False)
print(f'저장 완료: featurized_highk.csv')
print(f'shape: {df_final.shape}  (소재 수 x feature 수+3)')